In [ ]:
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
import torch
import json
from tqdm import tqdm

In [ ]:
model_name='granite'
model_path='/data/jfraile/llms/granite-3.3-2b-instruct'

In [4]:
model= AutoModelForCausalLM.from_pretrained(model_path, device_map='auto', torch_dtype=torch.bfloat16, trust_remote_code=True).eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]


In [ ]:
def interact_with_model(prompt, max_tokens=500):
    pipeline1 = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    )

    pipeline_kwargs = {
        "do_sample": True,
        "temperature": 0.75,
        "max_new_tokens": max_tokens,
        "return_full_text": False
    }
    chat=[{'role':'user', 'content':prompt}]
    final_prompt=tokenizer.apply_chat_template(chat, tokenize=False)
    pipeline_kwargs=pipeline_kwargs
    pipeline_kwargs['eos_token_id']=tokenizer.eos_token_id
    for out in pipeline1(final_prompt,**pipeline_kwargs):
        model_out=out['generated_text']
    return model_out




# Dipromats

In [ ]:
def load_annotations():
    path=f'/data/jfraile/Programs/datasets/DIPROMATS_2024_T2/dipromats24_t2_train_es.json'
    with open(path, 'r') as file:
        data=json.load(file)
        data=data['dataset']
        df_es=pd.DataFrame(data)

        country_columns = {
            "China": [f"CH{i}" for i in range(1, 7)],
            "Russia": [f"RU{i}" for i in range(1, 7)],
            "EU": [f"EU{i}" for i in range(1, 7)],
            "USA": [f"US{i}" for i in range(1, 7)],
        }

        for idx, row in df_es.iterrows():
            country = row["country"]
            country_narratives = country_columns.get(country, [])
            
            for i, narrative in enumerate(country_narratives, start=1):
                df_es.loc[idx, f"N{i}"] = row[narrative]

        df_es.drop(columns=[col for cols in country_columns.values() for col in cols], inplace=True)

    path=f'/data/jfraile/Programs/datasets/DIPROMATS_2024_T2/dipromats24_t2_train_en.json'
    with open(path, 'r') as file:
        data=json.load(file)
        narratives=data['narratives']
        data=data['dataset']
        df_en=pd.DataFrame(data)
        country_columns = {
            "China": [f"CH{i}" for i in range(1, 7)],
            "Russia": [f"RU{i}" for i in range(1, 7)],
            "EU": [f"EU{i}" for i in range(1, 7)],
            "USA": [f"US{i}" for i in range(1, 7)],
        }

        for idx, row in df_en.iterrows():
            country = row["country"]
            country_narratives = country_columns.get(country, [])
            
            for i, narrative in enumerate(country_narratives, start=1):
                df_en.loc[idx, f"N{i}"] = row[narrative]

        df_en.drop(columns=[col for cols in country_columns.values() for col in cols], inplace=True)
    
    df_train=pd.concat([df_es, df_en], ignore_index=True)
    columns=['tweet_id', 'text', 'country', 'N1', 'N2', 'N3', 'N4', 'N5', 'N6']
    df_train=df_train[columns]
    narratives=pd.DataFrame(narratives)
    narratives['country']=narratives['country'].map({'China':'China', 'Russia':'Russia', 'European Union':'EU', 'USA':'USA'})
    return df_train, narratives

def write_examples(examples):
    t="""
	
"""
    for ex in examples:
        ex=ex.replace("\n", " ")
        t=t+f"\t- {ex} \n"
    return t

In [7]:
df, narratives=load_annotations()

# Dipromats Description from titles

In [8]:
def get_prompt(country_id, title):
    if country_id=='USA':
        country='United States of America'
    elif country_id=='EU':
        country='European Union'
    else:
        country=country_id

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative about {country}. You also have to take into account the social, political and cultural context because the tweets were written throughout the year 2020. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in tweets.


Narrative title: {title} 

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [9]:
df_results=pd.DataFrame(columns=['narrative', 'result'])
for i in tqdm(range(len(narratives))):
    nar=narratives.iloc[i]
    country=nar['country']
    title=nar['title']
    n=nar['n_id'][-1]
    description=nar['description']
    dft=df[(df['country'] == country) & (df[f'N{n}'] != 'no')]
    examples=dft['text'].values
    prompt=get_prompt(country, title)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame({'narrative': [nar['n_id']], 'result': [result]})
    df_results = pd.concat([df_results, new_row], ignore_index=True)


    

100%|██████████| 24/24 [02:23<00:00,  5.96s/it]


In [10]:
df_results

,narrative,result
0,CH1,"Response: This narrative, presented in a serie..."
1,CH2,"Response:\n\nThis narrative, titled ""China is ..."
2,CH3,"\nResponse:\n\nThis narrative, ""China has an e..."
3,CH4,Response:\n\nThis narrative explores the persp...
4,CH5,"\nResponse:\n\nThis narrative, titled ""The gov..."
5,CH6,Response:\n\nThis narrative offers an intrigui...
6,EU1,"\nResponse: This narrative, titled ""The Europe..."
7,EU2,"Response:\n\nIn this narrative, the European U..."
8,EU3,Description: This narrative encapsulates the a...
9,EU4,response: This narrative explores the European...


In [11]:
name=f'z_s_{model_name}_only_title'
df_results.to_pickle(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.pkl')
df_results.to_json(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.json', orient='records', indent=2, force_ascii=False)

# Dipromats improved Description from title and Description

In [12]:
def get_prompt(country_id, title, description):
    if country_id=='USA':
        country='United States of America'
    elif country_id=='EU':
        country='European Union'
    else:
        country=country_id

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title and a paragraph with a description of a narrative about {country}. You also have to take into account the social, political and cultural context because the tweets were written throughout the year 2020. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in tweets.


Narrative title: {title} 
Description: {description}

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [13]:
df_results=pd.DataFrame(columns=['narrative', 'result'])
for i in tqdm(range(len(narratives))):
    nar=narratives.iloc[i]
    country=nar['country']
    title=nar['title']
    n=nar['n_id'][-1]
    description=nar['description']
    dft=df[(df['country'] == country) & (df[f'N{n}'] != 'no')]
    examples=dft['text'].values
    prompt=get_prompt(country, title, description)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame({'narrative': [nar['n_id']], 'result': [result]})
    df_results = pd.concat([df_results, new_row], ignore_index=True)


    

100%|██████████| 24/24 [02:43<00:00,  6.82s/it]


In [14]:
df_results

,narrative,result
0,CH1,\nImproved Description: During the tumultuous ...
1,CH2,\n[Response]\n\nThis narrative portrays China ...
2,CH3,"\nThe narrative ""China has an epic history,"" u..."
3,CH4,"\n responds: \n\nThroughout 2020, a series of ..."
4,CH5,"\nResponse: This narrative, emerging in the co..."
5,CH6,Responding:\n\nThis narrative unfolds as a vib...
6,EU1,\nResponse: This narrative presents a compelli...
7,EU2,"\nResponse: In 2020, the European Union's stre..."
8,EU3,\nThe tweet narrative advocates for the Europe...
9,EU4,"\nThe European Union, in its dynamic and trans..."


In [15]:
name=f'z_s_{model_name}_description'
df_results.to_pickle(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.pkl')
df_results.to_json(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.json', orient='records', indent=2, force_ascii=False)

# Dipromats completed Description

In [16]:
def get_prompt(country_id, title, description):
    if country_id=='USA':
        country='United States of America'
    elif country_id=='EU':
        country='European Union'
    else:
        country=country_id

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive a paragraph with a description of a narrative about {country}. 
You also have to take into account the social, political and cultural context because the tweets were written throughout the year 2020. 
Your mission will be to rewrite the paragraph to improve the readability and interpretation of that text. Your description will have to help improve the identification of narratives in tweets. 
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the narrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 


Narrative title: {title} 
Description: {description}

    """
    return user_prompt

In [17]:
df_results=pd.DataFrame(columns=['narrative', 'result'])
for i in tqdm(range(len(narratives))):
    nar=narratives.iloc[i]
    country=nar['country']
    title=nar['title']
    n=nar['n_id'][-1]
    description=nar['description']
    dft=df[(df['country'] == country) & (df[f'N{n}'] != 'no')]
    examples=dft['text'].values
    prompt=get_prompt(country, title, description)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame({'narrative': [nar['n_id']], 'result': [result]})
    df_results = pd.concat([df_results, new_row], ignore_index=True)


    

100%|██████████| 24/24 [05:03<00:00, 12.67s/it]


In [18]:
df_results

,narrative,result
0,CH1,[Answer]\n\n# Improved Prompt:\n\nTitle: Analy...
1,CH2,[Answer]\n\nNarrative Title: China's Global Le...
2,CH3,[Response]\n\nTitle: China's Unyielding Histor...
3,CH4,[Response]\n\nNarrative Title: Celebrating Chi...
4,CH5,"[Response]\n\nNarrative Title: ""China's Centra..."
5,CH6,"[Response]:\n\nTitle: ""Exploring the Rich Tape..."
6,EU1,[Response]\n\nNarrative Title: Triumphs of the...
7,EU2,[Revised Prompt]\n\nYour task is to refine the...
8,EU3,"[Response]\n\nTitle: ""Empowering European Citi..."
9,EU4,Response:\n\nNarrative Title: The EU Leads the...


In [19]:
name=f'z_s_{model_name}_description_complete'
df_results.to_pickle(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.pkl')
df_results.to_json(f'/data/jfraile/Programs/Narratives_Identification/narratives_descriptions/zero_shot_description/{name}.json', orient='records', indent=2, force_ascii=False)